In [5]:
import torch
import cv2
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from model.efficientnet_lstm import efficientnet_lstm
from model.efficientnet_transformer import efficientnet_transformer
from inference.inference_dataset import InferenceDataset, ManipulationDataset
from utils.grad_cam import GradCAM, overlay_heatmap, compute_gradcam
from utils.checkpoints import load_checkpoint

## Model

In [4]:
device = "cuda"

model_efficientnet_lstm_pretrained = efficientnet_lstm(pretrained=False, num_classes=2).to(device)
# model_efficientnet_transformer = efficientnet_transformer(pretrained=False, num_classes=2).to(device)

model_efficientnet_lstm_pretrained, _, _ = load_checkpoint(
    ckpt_path = "D:/DS/projects/deepfake_detection_0_0_2/checkpoints/efficientnet_lstm/epoch_11.pth",
    model=model_efficientnet_lstm_pretrained,
    optimizer=None,
    device=device,
    load_opt=False
)
model_efficientnet_lstm_pretrained.eval()

[INFO] Loaded checkpoint from D:/DS/projects/deepfake_detection_0_0_2/checkpoints/efficientnet_lstm/epoch_11.pth (epoch 12)


efficientnet_lstm(
  (backbone): EfficientNet(
    (_conv_stem): Conv2dStaticSamePadding(
      3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False
      (static_padding): ZeroPad2d((0, 1, 0, 1))
    )
    (_bn0): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
    (_blocks): ModuleList(
      (0): MBConvBlock(
        (_depthwise_conv): Conv2dStaticSamePadding(
          32, 32, kernel_size=(3, 3), stride=[1, 1], groups=32, bias=False
          (static_padding): ZeroPad2d((1, 1, 1, 1))
        )
        (_bn1): BatchNorm2d(32, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
        (_se_reduce): Conv2dStaticSamePadding(
          32, 8, kernel_size=(1, 1), stride=(1, 1)
          (static_padding): Identity()
        )
        (_se_expand): Conv2dStaticSamePadding(
          8, 32, kernel_size=(1, 1), stride=(1, 1)
          (static_padding): Identity()
        )
        (_project_conv): Conv2dStaticSame

## Dataset & Dataloader

In [3]:
testset = InferenceDataset()
testloader = DataLoader(testset, batch_size=1, shuffle=False)

preds = []
labels = []

Indexing test: 100%|██████████| 140/140 [00:00<00:00, 191.92it/s]


[INFO] Tổng số video: 700 (phase=test)
[INFO] Real: 140 | Fake: 560


In [4]:
test_set = ManipulationDataset(dataset=['Deepfakes'])
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

preds = []
labels = []

[INFO] Loaded 280 videos (140 fakes, 140 reals)


## Inference

In [10]:
with torch.no_grad():
        for batch in tqdm(testloader, desc="Evaluating"):
            clip = batch["clip"].to(device)     # (1, C, T, H, W)
            label = batch["label"].item()

            outputs = model_efficientnet_lstm_pretrained(clip)
            logits = outputs["cls"] if isinstance(outputs, dict) else outputs

            prob_fake = torch.softmax(logits, dim=1)[0, 1].item()

            preds.append(prob_fake)
            labels.append(label)

auc = roc_auc_score(labels, preds)
print(f"\n[INFO] AUC = {auc:.4f}")

Evaluating: 100%|██████████| 700/700 [02:48<00:00,  4.16it/s]


[INFO] AUC = 0.9906


In [5]:
with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            clip = batch["clip"].to(device)     # (1, C, T, H, W)
            label = batch["label"].item()

            outputs = model_efficientnet_lstm_pretrained(clip)
            logits = outputs["cls"] if isinstance(outputs, dict) else outputs

            prob_fake = torch.softmax(logits, dim=1)[0, 1].item()

            preds.append(prob_fake)
            labels.append(label)

auc = roc_auc_score(labels, preds)
print(f"\n[INFO] AUC = {auc:.4f}")

Evaluating: 100%|██████████| 280/280 [01:07<00:00,  4.17it/s]


[INFO] AUC = 0.9963
